In [1]:
from google.colab import files
uploaded=files.upload()

Saving transactions.csv to transactions.csv
Saving settlements.csv to settlements.csv
Saving refunds.csv to refunds.csv
Saving fees.csv to fees.csv


In [2]:
import pandas as pd
import numpy as np

print("LedgerLens Day 2 started!")

LedgerLens Day 2 started!


In [3]:
transactions = pd.read_csv("transactions.csv")
fees = pd.read_csv("fees.csv")
refunds = pd.read_csv("refunds.csv")
settlements = pd.read_csv("settlements.csv")

print("Transactions :", len(transactions))
print("Fees        :", len(fees))
print("Refunds     :", len(refunds))
print("Settlements :", len(settlements))

Transactions : 2000
Fees        : 2000
Refunds     : 152
Settlements : 1947


In [4]:
print("TRANSACTIONS")
display(transactions.head())

print("\nFEES")
display(fees.head())

print("\nREFUNDS")
display(refunds.head())

print("\nSETTLEMENTS")
display(settlements.head())

TRANSACTIONS


,transaction_id,order_id,customer_id,amount,payment_date,payment_status
0,TXN000001,ORD000001,CUST00655,2872.14,2026-08-24,SUCCESS
1,TXN000002,ORD000002,CUST00282,6197.81,2026-08-05,SUCCESS
2,TXN000003,ORD000003,CUST00105,16949.82,2026-08-18,SUCCESS
3,TXN000004,ORD000004,CUST00090,14803.26,2026-08-02,SUCCESS
4,TXN000005,ORD000005,CUST00031,2433.01,2026-08-08,SUCCESS



FEES


,transaction_id,gateway_fee,tax_on_fee
0,TXN000001,32.58,5.86
1,TXN000002,124.43,22.40
2,TXN000003,277.18,49.89
3,TXN000004,225.37,40.57
4,TXN000005,36.04,6.49



REFUNDS


,refund_id,transaction_id,refund_amount,refund_status
0,REF000001,TXN000002,2526.04,SUCCESS
1,REF000002,TXN000020,4159.84,SUCCESS
2,REF000003,TXN000021,1883.08,SUCCESS
3,REF000004,TXN000052,1543.99,SUCCESS
4,REF000005,TXN000068,340.07,SUCCESS



SETTLEMENTS


,settlement_id,transaction_id,settled_amount,settlement_date
0,SET000001,TXN000001,2833.70,2026-08-26
1,SET000002,TXN000002,3524.94,2026-08-08
2,SET000003,TXN000003,16622.75,2026-08-19
3,SET000004,TXN000004,14537.32,2026-08-03
4,SET000005,TXN000005,2390.48,2026-08-09


In [5]:
def build_reconciliation(
    transactions,
    fees,
    refunds,
    settlements
):

    # -----------------------------------------
    # 1. Merge transaction + fee information
    # -----------------------------------------

    df = transactions.merge(
        fees,
        on="transaction_id",
        how="left"
    )

    # -----------------------------------------
    # 2. Calculate total refund per transaction
    # -----------------------------------------

    if len(refunds) > 0:

        refund_summary = (
            refunds
            .groupby("transaction_id")["refund_amount"]
            .sum()
            .reset_index()
        )

    else:

        refund_summary = pd.DataFrame(
            columns=[
                "transaction_id",
                "refund_amount"
            ]
        )

    df = df.merge(
        refund_summary,
        on="transaction_id",
        how="left"
    )

    df["refund_amount"] = (
        df["refund_amount"]
        .fillna(0)
    )

    # -----------------------------------------
    # 3. Calculate expected settlement
    # -----------------------------------------

    df["expected_settlement"] = (
        df["amount"]
        - df["gateway_fee"]
        - df["tax_on_fee"]
        - df["refund_amount"]
    )

    df["expected_settlement"] = (
        df["expected_settlement"]
        .clip(lower=0)
        .round(2)
    )

    # -----------------------------------------
    # 4. Add actual settlement
    # -----------------------------------------

    settlement_info = settlements[
        [
            "transaction_id",
            "settlement_id",
            "settled_amount",
            "settlement_date"
        ]
    ]

    df = df.merge(
        settlement_info,
        on="transaction_id",
        how="left"
    )

    # -----------------------------------------
    # 5. Calculate discrepancy
    # -----------------------------------------

    df["difference"] = (
        df["expected_settlement"]
        - df["settled_amount"]
    )

    df["difference"] = (
        df["difference"]
        .fillna(df["expected_settlement"])
        .round(2)
    )

    # -----------------------------------------
    # 6. Classify exception
    # -----------------------------------------

    def classify(row):

        if pd.isna(row["settled_amount"]):
            return "MISSING_SETTLEMENT"

        if abs(row["difference"]) <= 1:
            return "MATCHED"

        if row["difference"] > 1:
            return "UNDER_SETTLED"

        return "OVER_SETTLED"

    df["status"] = df.apply(
        classify,
        axis=1
    )

    return df

In [6]:
reconciliation = build_reconciliation(
    transactions,
    fees,
    refunds,
    settlements
)

display(
    reconciliation.head(10)
)

,transaction_id,order_id,customer_id,amount,payment_date,payment_status,gateway_fee,tax_on_fee,refund_amount,expected_settlement,settlement_id,settled_amount,settlement_date,difference,status
0,TXN000001,ORD000001,CUST00655,2872.14,2026-08-24,SUCCESS,32.58,5.86,0.00,2833.70,SET000001,2833.70,2026-08-26,0.0,MATCHED
1,TXN000002,ORD000002,CUST00282,6197.81,2026-08-05,SUCCESS,124.43,22.40,2526.04,3524.94,SET000002,3524.94,2026-08-08,0.0,MATCHED
2,TXN000003,ORD000003,CUST00105,16949.82,2026-08-18,SUCCESS,277.18,49.89,0.00,16622.75,SET000003,16622.75,2026-08-19,0.0,MATCHED
3,TXN000004,ORD000004,CUST00090,14803.26,2026-08-02,SUCCESS,225.37,40.57,0.00,14537.32,SET000004,14537.32,2026-08-03,0.0,MATCHED
4,TXN000005,ORD000005,CUST00031,2433.01,2026-08-08,SUCCESS,36.04,6.49,0.00,2390.48,SET000005,2390.48,2026-08-09,0.0,MATCHED
5,TXN000006,ORD000006,CUST00518,15090.27,2026-08-18,SUCCESS,285.33,51.36,0.00,14753.58,SET000006,14753.58,2026-08-20,0.0,MATCHED
6,TXN000007,ORD000007,CUST00204,17928.89,2026-08-23,SUCCESS,185.80,33.44,0.00,17709.65,SET000007,17709.65,2026-08-25,0.0,MATCHED
7,TXN000008,ORD000008,CUST00559,10546.04,2026-08-15,SUCCESS,153.68,27.66,0.00,10364.70,SET000008,10364.70,2026-08-16,0.0,MATCHED
8,TXN000009,ORD000009,CUST00604,7026.95,2026-08-01,SUCCESS,174.37,31.39,0.00,6821.19,SET000009,6821.19,2026-08-04,0.0,MATCHED
9,TXN000010,ORD000010,CUST00164,17483.67,2026-08-11,SUCCESS,336.44,60.56,0.00,17086.67,SET000010,17086.67,2026-08-12,0.0,MATCHED


In [7]:
total_transactions = len(reconciliation)

matched = (
    reconciliation["status"] == "MATCHED"
).sum()

exceptions = (
    reconciliation["status"] != "MATCHED"
).sum()

match_rate = (
    matched / total_transactions
) * 100

exception_rate = (
    exceptions / total_transactions
) * 100

total_expected = (
    reconciliation["expected_settlement"]
    .sum()
)

total_actual = (
    reconciliation["settled_amount"]
    .fillna(0)
    .sum()
)

total_unexplained = (
    reconciliation.loc[
        reconciliation["status"] != "MATCHED",
        "difference"
    ]
    .sum()
)

print("==========================================")
print("       LEDGERLENS RECONCILIATION")
print("==========================================")

print(f"Total Transactions : {total_transactions:,}")
print(f"Matched            : {matched:,}")
print(f"Exceptions         : {exceptions:,}")
print(f"Match Rate         : {match_rate:.2f}%")
print(f"Exception Rate     : {exception_rate:.2f}%")

print(
    f"\nExpected Settlement : ₹{total_expected:,.2f}"
)

print(
    f"Actual Settlement   : ₹{total_actual:,.2f}"
)

print(
    f"Unexplained Amount  : ₹{total_unexplained:,.2f}"
)

       LEDGERLENS RECONCILIATION
Total Transactions : 2,000
Matched            : 1,894
Exceptions         : 106
Match Rate         : 94.70%
Exception Rate     : 5.30%

Expected Settlement : ₹22,981,731.29
Actual Settlement   : ₹22,359,792.31
Unexplained Amount  : ₹621,938.98


In [8]:
exception_breakdown = (
    reconciliation["status"]
    .value_counts()
    .reset_index()
)

exception_breakdown.columns = [
    "status",
    "count"
]

display(exception_breakdown)

,status,count
0,MATCHED,1894
1,UNDER_SETTLED,53
2,MISSING_SETTLEMENT,53


In [9]:
def calculate_priority(row):

    difference = abs(row["difference"])

    anomaly_base = difference

    if row["status"] == "MISSING_SETTLEMENT":
        anomaly_base += 500

    elif row["status"] == "UNDER_SETTLED":
        anomaly_base += 300

    elif row["status"] == "OVER_SETTLED":
        anomaly_base += 100

    return anomaly_base


reconciliation["priority_score"] = (
    reconciliation.apply(
        calculate_priority,
        axis=1
    )
)

In [10]:
def priority_level(score):

    if score >= 5000:
        return "CRITICAL"

    elif score >= 2000:
        return "HIGH"

    elif score >= 500:
        return "MEDIUM"

    return "LOW"


reconciliation["priority"] = (
    reconciliation["priority_score"]
    .apply(priority_level)
)

In [11]:
top_exceptions = (
    reconciliation[
        reconciliation["status"] != "MATCHED"
    ]
    .sort_values(
        "priority_score",
        ascending=False
    )
    .head(20)
)

display(
    top_exceptions[
        [
            "transaction_id",
            "amount",
            "expected_settlement",
            "settled_amount",
            "difference",
            "status",
            "priority",
            "priority_score"
        ]
    ]
)

,transaction_id,amount,expected_settlement,settled_amount,difference,status,priority,priority_score
1594,TXN001595,23710.47,23100.34,NaN,23100.34,MISSING_SETTLEMENT,CRITICAL,23600.34
1740,TXN001741,23274.06,22624.64,NaN,22624.64,MISSING_SETTLEMENT,CRITICAL,23124.64
1460,TXN001461,22629.01,22319.86,NaN,22319.86,MISSING_SETTLEMENT,CRITICAL,22819.86
1282,TXN001283,22056.09,21677.18,NaN,21677.18,MISSING_SETTLEMENT,CRITICAL,22177.18
783,TXN000784,22067.90,21434.24,NaN,21434.24,MISSING_SETTLEMENT,CRITICAL,21934.24
1895,TXN001896,21605.08,21249.27,NaN,21249.27,MISSING_SETTLEMENT,CRITICAL,21749.27
359,TXN000360,21497.62,21060.74,NaN,21060.74,MISSING_SETTLEMENT,CRITICAL,21560.74
1693,TXN001694,20550.45,20216.36,NaN,20216.36,MISSING_SETTLEMENT,CRITICAL,20716.36
866,TXN000867,20163.83,19687.74,NaN,19687.74,MISSING_SETTLEMENT,CRITICAL,20187.74
1172,TXN001173,19360.83,19034.62,NaN,19034.62,MISSING_SETTLEMENT,CRITICAL,19534.62


In [12]:
exceptions_report = reconciliation[
    reconciliation["status"] != "MATCHED"
].copy()

exceptions_report = exceptions_report.sort_values(
    "priority_score",
    ascending=False
)

exceptions_report.to_csv(
    "reconciliation_exceptions.csv",
    index=False
)

print(
    f"Exception report created: "
    f"{len(exceptions_report)} records"
)

Exception report created: 106 records


In [13]:
reconciliation.to_csv(
    "reconciliation_report_day2.csv",
    index=False
)

print("Complete reconciliation report created!")

Complete reconciliation report created!


In [14]:
evaluation = pd.DataFrame({
    "metric": [
        "Total Transactions",
        "Matched Transactions",
        "Exceptions",
        "Match Rate",
        "Exception Rate",
        "Expected Settlement",
        "Actual Settlement",
        "Unexplained Amount"
    ],

    "value": [
        total_transactions,
        matched,
        exceptions,
        round(match_rate, 2),
        round(exception_rate, 2),
        round(total_expected, 2),
        round(total_actual, 2),
        round(total_unexplained, 2)
    ]
})

display(evaluation)

,metric,value
0,Total Transactions,2000.00
1,Matched Transactions,1894.00
2,Exceptions,106.00
3,Match Rate,94.70
4,Exception Rate,5.30
5,Expected Settlement,22981731.29
6,Actual Settlement,22359792.31
7,Unexplained Amount,621938.98


In [15]:
evaluation.to_csv(
    "day2_evaluation.csv",
    index=False
)

print("Day 2 evaluation saved!")

Day 2 evaluation saved!
